## 2. 벡터 저장소 (Vector Store)

In [ ]:

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

In [7]:
from langchain_elasticsearch import ElasticsearchStore

vector_poi = "vector_poi"

vector_store = ElasticsearchStore(
    embedding=embeddings_model,
    index_name=vector_poi,
    es_url="http://localhost:9200"
)

In [ ]:
import pandas as pd

fielePath = "./data/poi_data.csv"

def read_excel_as_list(file_path):
    # Excel 파일 읽기
    try:
        # pandas로 Excel 파일 읽기
        df = pd.read_csv(file_path, encoding='EUC-KR')
        
        # DataFrame을 리스트로 변환
        data_list = df.values.tolist()

        return data_list

    except Exception as e:
        print(f"Error reading the Excel file: {e}")
        return []

data_list = read_excel_as_list(file_path=fielePath)
print(data_list[0])
print(len(data_list))

In [5]:
import math

a    = 6378137.0               
f    = 1 / 298.257222101      
lat0 = 38.0 * math.pi / 180.0  
lon0 = 127.0 * math.pi / 180.0 
k0   = 1.0                     
x0   = 200000.0                
y0   = 500000.0   

def tm_to_wgs84(x, y):

    math.sqrt(2 * f- f * f)
    n = f / (2 - f)
    A = a / (1 + n) * (1 + n*n/4 + n*n*n*n/64)

    x = x - x0
    y = y - y0

    lat = lat0 
    for i in range(5):
        lat = (y / (k0 * A)) + lat0
	
    lon = lon0 + (x / (k0 * A * math.cos(lat)))
    lat_deg = lat * 180.0 / math.pi
    lon_deg = lon * 180.0 / math.pi

    return lon_deg, lat_deg

# embedding field using embedded model 


In [16]:
# declare variance
es_url = "http://localhost:9200"
es_index_name = "vector_poi"

mapping = {
    "mappings": {
        "properties": {
            "title_vector": {  # title 벡터 필드
                "type": "dense_vector",
                "dims": 1024  # Hugging Face 모델에서 생성된 벡터 차원 수
            },
            "address_vector": {  # address 벡터 필드
                "type": "dense_vector",
                "dims": 1024
            },
            "location": {  # lat, lon을 포함하는 geo_point 필드
                "type": "geo_point"
            },
            "title": {  # title 원본 텍스트
                "type": "text"
            },
            "address": {  # address 원본 텍스트
                "type": "text"
            }
        }
    }
}

template = {
    "title": "{{title}}",
    "address": "{{address}}",
    "location": {
        "lat": "{{lat}}",
        "lon": "{{lon}}"
    },
    "vector": "{{vector}}"
}



In [19]:
from elasticsearch import Elasticsearch

# Elasticsearch 클라이언트 초기화
es = Elasticsearch("http://localhost:9200")


# 인덱스 생성
if not es.indices.exists(index=es_index_name):
    es.indices.create(index=es_index_name, body=mapping)
    print(f"Elasticsearch 인덱스 '{es_index_name}'가 생성되었습니다.")
else:
    print(f"Elasticsearch 인덱스 '{es_index_name}'는 이미 존재합니다.")

Elasticsearch 인덱스 'vector_poi'는 이미 존재합니다.


In [ ]:
bulk_data = []
for doc in data_list:
    lon, lat = tm_to_wgs84(doc[4], doc[5])
    print(doc[3], doc[9])
    title_vector = embeddings_model.embed_query(doc[3])
    address_vector = embeddings_model.embed_query(doc[9])
    source = {
         "_index": es_index_name, 
        "_source": {
            "title": doc[3],
            "address": doc[9],
            "title_vector": title_vector,
            "address_vector": address_vector,
            "location": {"lat": lat, "lon": lon}
        }
    }
    bulk_data.append(source)


    

In [ ]:
from elasticsearch import helpers
import json

vector = [item['_source']['title_vector'] for item in bulk_data]
print(len(vector[0]))

helpers.bulk(es, bulk_data)
print("inserting bulk is complete")

## 검색

In [ ]:
query_text = "종로"  # 검색할 텍스트
query_embedding = embeddings_model.embed_query(query_text)  # 쿼리를 벡터화
# Elasticsearch 검색 요청
search_query = {
    "_source" : ["title", "address", "location"],
    "query": {
        "script_score": {
            "query": {
                "match_all": {}  # 모든 문서에서 스코어 기반 필터링
            },
            "script": {
                "source": """
                    cosineSimilarity(params.query_vector, 'title_vector') + 
                    cosineSimilarity(params.query_vector, 'address_vector')
                """,
                "params": {
                    "query_vector": query_embedding
                }
            }
        }
    }
}
print(search_query)


# 검색 요청 실행
response = es.search(index=es_index_name, body=search_query)

# 검색 결과 출력
print("벡터 검색 결과:")
for hit in response["hits"]["hits"]:
    print(hit["_source"])



# 랭체인 fastAPI 임시 대용

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import uvicorn
import nest_asyncio


app = FastAPI()

class QueryRequest(BaseModel):
    query: str


@app.post("/embed/query")
def get_embedding(request: QueryRequest):
    query_text = request.query

    if not query_text:
        raise HTTPException(status_code=400, detail="Query is empty.")

    # query를 벡터로 변환
    vector = embeddings_model.embed_query(query_text)

    # 결과 반환 (JSON)
    return {"vector": vector}

@app.get("/health")
def get_health_check():
    return {"status" : "up"}

nest_asyncio.apply()
uvicorn.run(app)
